# 08b · INT8 ablation analysis (no GPU)

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 1 (follow-up to the pilot review; reads only)

**Purpose.** Evaluate every ablation engine under four threshold rules and compare the seed ranges of each setting with the natural setting.

**In plain words.** For each engine we read the saved dev and test video scores. Thresholds come only from the development set: A is the fixed FP32 threshold, B the engine's own EER point (kept for comparison), C the engine's own threshold that rejects 10% or 15% of real dev faces. A setting probably differs from natural when its range over the 3 seeds does not overlap natural's range.

**Expected inputs**
- Notebook 08 outputs: `results/int8/ablation/int8_entropy_<setting>_s<seed>/`; FP32 results `results/fp32/`; optionally the pilot results `results/int8/int8_entropy_natural_s<seed>/`.

**Expected outputs**
- `QPAD_Project/results/int8/ablation/analysis/` (`ablation_metrics.csv`, `ablation_ranges.csv`, `ablation_thresholds.csv`, `ablation_score_shifts.csv`, `pilot_vs_ablation_natural.csv`, and one figure per ablation line: `ablation_attack_share.png/.pdf`, `ablation_images.png/.pdf`)

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42

if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb08b", paths.logs / "nb08b.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

In [ ]:
import json, pandas as pd
from qpad import ablation as AB
from qpad.experiments import engine_tag
from qpad.pipeline import find_input
icfg, acfg = cfgs["int8"], cfgs["int8"]["ablation"]
METHOD = icfg["calibration_method"]["primary"]
TARGETS = [float(t) for t in acfg["bpcer_targets"]]
BASELINE = acfg["baseline"]
SETTINGS_TABLE = AB.settings_table(acfg, int(icfg["calibration_budget"]["frames_per_video"]),
                                   int(icfg["calibration_method"]["calibration_batch_size"]))
RESULT_DIRS, MISSING = {}, []
for setting in SETTINGS_TABLE["setting"]:
    for seed in acfg["seeds"]:
        d = find_input(paths, DRY_RUN, "results", f"int8/ablation/{engine_tag(METHOD, setting, seed)}")
        if (d / "metrics.json").is_file():
            RESULT_DIRS[(setting, seed)] = d
        else:
            MISSING.append(f"{setting} s{seed}")
print(f"engines with results: {len(RESULT_DIRS)}; missing: {MISSING or 'none'}")
if not RESULT_DIRS:
    raise FileNotFoundError("no ablation results found; run notebook 08 first")
FP32_DIR = find_input(paths, DRY_RUN, "results", "fp32")
ANALYSIS = OUT.results / "int8" / "ablation" / "analysis"
ANALYSIS.mkdir(parents=True, exist_ok=True)
print("writing to:", ANALYSIS)

## 1. Threshold rules
Each engine's thresholds under every rule, with the range over seeds. The pilot's EER thresholds (rule B) jumped between 0.51 and 0.76; the C rules use the 27th or 40th lowest real-face dev score instead of one borderline video, so they should move much less.

In [ ]:
LONG = AB.collect(RESULT_DIRS, TARGETS)
LONG.to_csv(ANALYSIS / "ablation_metrics.csv", index=False)
fp32_thr = json.loads((FP32_DIR / "threshold.json").read_text())["threshold"]
FP32_RULES = AB.rule_metrics(AB.read_videos(FP32_DIR, "dev"), AB.read_videos(FP32_DIR, "test"),
                             {"A_fp32": fp32_thr, "B_dev_eer": fp32_thr}, TARGETS)
print("FP32 reference (same rules on the FP32 model):")
display(FP32_RULES[["rule", "threshold", "print_apcer_errors", "replay_apcer_errors", "bpcer_errors", "acer_pct"]])
THRESHOLDS = LONG.pivot_table(index=["setting", "seed"], columns="rule", values="threshold")
THRESHOLDS.to_csv(ANALYSIS / "ablation_thresholds.csv")
display(THRESHOLDS.round(4))
print("Spread of the threshold over all engines (max - min):", (THRESHOLDS.max() - THRESHOLDS.min()).round(4).to_dict())

## 2. Seed ranges per setting
Min–max over seeds of the test error rates (%) under rules A, C10 and C15. `*` marks a range that does not overlap the natural setting's range. With 3 seeds this is a screening rule, not a significance test.

In [ ]:
RANGES = AB.setting_ranges(LONG, BASELINE)
RANGES.to_csv(ANALYSIS / "ablation_ranges.csv", index=False)
MAIN_RULES = ["A_fp32", *[AB.bpcer_rule(t) for t in TARGETS]]
for rule in MAIN_RULES + ["B_dev_eer"]:
    print(AB.RULE_LABELS.get(rule, rule))
    display(AB.range_text(RANGES, [rule], ["print_apcer_pct", "replay_apcer_pct", "bpcer_pct"])
            .reindex(SETTINGS_TABLE["setting"]).dropna(how="all"))

## 3. Figures: where do results start to get worse?

In [ ]:
for line, levels in acfg["lines"].items():
    present = [s for s in levels if any(k[0] == s for k in RESULT_DIRS)]
    if present:
        print(AB.fig_ablation_line(LONG, SETTINGS_TABLE, line, present, MAIN_RULES, ANALYSIS, f"ablation_{line}")["png"])

## 4. Score changes against natural (no threshold)
Mean change of the test video scores against the natural engine of the same seed. A positive mean on attacks means attacks look more like real faces. This does not depend on any threshold.

In [ ]:
SHIFTS = AB.score_shifts_vs_baseline(RESULT_DIRS, BASELINE)
if len(SHIFTS):
    SHIFTS.to_csv(ANALYSIS / "ablation_score_shifts.csv", index=False)
    display(SHIFTS.groupby(["setting", "class"])[["mean_delta", "mean_abs_delta", "max_abs_delta"]]
            .agg(["min", "max"]).round(4).reindex(SETTINGS_TABLE["setting"], level=0))
else:
    print("only the baseline is available (dry run)")

## 5. Natural engines of this run against the pilot's
Natural seeds 1 and 2 use exactly the pilot's calibration images, so these pairs are rebuilds. Their difference is the noise floor that any ablation effect has to exceed. If the ablation ran on a different GPU than the pilot, the difference also includes the GPU change.

In [ ]:
from qpad import diagnostics as D
PAIRS = []
for seed in acfg["seeds"]:
    tag = engine_tag(METHOD, BASELINE, seed)
    try:   # the pilot built only some seeds (natural s1, s2)
        pilot = find_input(paths, DRY_RUN, "results", f"int8/{tag}")
    except FileNotFoundError:
        continue
    if (pilot / "metrics.json").is_file() and (BASELINE, seed) in RESULT_DIRS:
        metas = [find_input(paths, DRY_RUN, "engines", t) / "engine_meta.json" for t in (tag, f"ablation/{tag}")]
        gpus = sorted({json.loads(m.read_text())["hardware"]["gpu_name"] for m in metas if m.is_file()})
        note = f" (different GPUs: {' vs '.join(gpus)})" if len(gpus) > 1 else ""
        PAIRS.append((f"pilot vs ablation natural s{seed}{note}", pilot, RESULT_DIRS[(BASELINE, seed)]))
if PAIRS:
    SHIFT_P, FLIPS_P = D.compare_pairs(PAIRS, "A")
    SHIFT_P.to_csv(ANALYSIS / "pilot_vs_ablation_natural.csv", index=False)
    display(SHIFT_P)
    display(FLIPS_P)
else:
    print("no pilot engine with the same seed found")

In [ ]:
CLEAR = RANGES[RANGES["separated"] & RANGES["rule"].isin(MAIN_RULES)
               & RANGES["metric"].isin(["print_apcer_pct", "replay_apcer_pct", "bpcer_pct"])]
print("Settings whose seed range is clear of natural's (rules A and C):")
print(CLEAR[["rule", "setting", "metric", "min", "max", "baseline_min", "baseline_max", "direction"]]
      .round(2).to_string(index=False) if len(CLEAR) else "none")

This is the last notebook of the ablation.